# Homework 2: Generative Models for Inverse Problems

This homework is based on the material of the **generative-model** chapter, with emphasis on the use of learned priors for inverse problems. The goal is to reuse pretrained generative models from the lecture notebooks and study how they can guide reconstruction under a known forward operator, following the broad perspective of learned priors surveyed in {cite}`ongie2020deep,arridge2019solving`.

Starting from clean Mayo images $\{\boldsymbol{x}_i\}_{i=1}^N$, you will generate synthetic measurements of the form

$$
    \boldsymbol{y}_i^\delta = K\boldsymbol{x}_i + \boldsymbol{e},
$$

where $K$ is a known motion-blur operator and $\boldsymbol{e}$ is additive Gaussian noise. You will then compare at least two reconstruction strategies:

- a **generator-range prior** method based on a pretrained **VAE decoder** {cite}`kingma2014auto,bora2017compressed`;
- a **diffusion-prior** method based on the pretrained denoiser from the diffusion notebook {cite}`ho2020denoising,chung2023diffusion`.

The purpose of the assignment is not to reproduce a research paper line by line, but to demonstrate that you understand the algorithmic logic of generative priors: latent-space optimization, data fidelity, denoising guidance, and the tradeoff between realism and measurement consistency.


## Homework Goals and Deliverables

You are asked to complete the notebook and submit the following:

1. A completed version of this notebook with all `TODO` sections filled in.
2. The pretrained weights you reused from the lecture notebooks, together with any additional saved results you produced for this homework.
3. A short written discussion, included at the end of the notebook, answering the conceptual questions.
4. At least one figure comparing the corrupted datum and the final reconstructions produced by your methods on the same test image.

Your work should show that you can:

- build a dataset pipeline consistent with the generative models used in the lectures;
- simulate an inverse problem through a known operator $K$;
- load and reuse pretrained generative models correctly;
- implement a latent-prior reconstruction procedure;
- implement a diffusion-prior reconstruction procedure;
- compare the methods critically rather than only visually.

```{note}
Use grayscale images resized to $64 \times 64$, because this is the resolution used by the VAE, GAN, and diffusion lecture notebooks.
```

```{warning}
Do **not** use external pretrained foundation models, external inverse-problem toolkits, or black-box diffusion pipelines. This includes downloading a ready-made Hugging Face diffusion pipeline instead of reusing the course denoiser. The homework is about reusing and understanding the models developed in this course.
```


## Suggested Structure of the Work

A reasonable workflow is to start from the data and corruption model, then load the pretrained generative models, then implement the reconstruction procedures, and finally compare them on the same examples.

The **mandatory** part of the homework is:

- one **VAE-based latent prior** reconstruction {cite}`kingma2014auto,bora2017compressed`;
- one **diffusion-prior** reconstruction, choosing either a DPS-style or a DiffPIR-style method {cite}`chung2023diffusion,zhu2023diffpir`.

An optional extension is to include a **GAN-based generator prior** or to implement **both** DPS-style and DiffPIR-style diffusion reconstructions and discuss how the comparison changes.

The notebook intentionally leaves several implementation choices open. You are expected to reuse the ideas developed in the lecture notebooks, but not simply copy them without understanding the role of each step.


In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import os
import sys
from pathlib import Path

if IN_COLAB:
    print("Configuring Google Colab environment...")
    
    # Mount Google Drive to access the dataset
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Unzip the dataset from Drive to Colab's fast local storage
    if not os.path.exists('/content/Mayo'):
        print("Extracting dataset from Google Drive...")
        os.system('unzip -q /content/drive/MyDrive/homeworks_CI/Mayo.zip -d /content/')
    
    # Install dependencies and clone repo
    os.system('pip install astra-toolbox')
    if not os.path.exists('CI_homeworks'):
        os.system('git clone -b hm3 https://github.com/alessandrocampedelli/CI_homeworks.git')
    sys.path.append('CI_homeworks')
    
    # Path to the dataset in fast local storage
    mayo_dataset_path = '/content/Mayo'
    book_root = Path('CI_homeworks').resolve()
    weights_dir = Path('/content/drive/MyDrive/homeworks_CI/weights2')
else:
    # Local setup
    sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
    book_root = Path('..').resolve()
    mayo_dataset_path = str(book_root / 'hm2' / 'Mayo')
    weights_dir = book_root / 'weights2'

weights_dir.mkdir(parents=True, exist_ok=True)

import glob
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

from IPPy import operators, utilities
from IPPy.utilities import gaussian_noise, get_device
from IPPy.utilities import metrics
from IPPy.nn.diffusion import DiffusionUNet, cosine_beta_schedule, extract, denormalize_to_01
from IPPy.nn.vae import ConvVAE


device = get_device()
torch.manual_seed(0)

print('Working device:', device)
print('Weights directory:', weights_dir)






## Part 1: Data Pipeline, Corruption Model, and Pretrained Models

In this first part, build a dataset for the Mayo images at $64 \times 64$, define the synthetic inverse problem, and load the pretrained generative models from the lecture notebooks.

The objective of this part is to verify that the data pipeline, the corruption model, and the reused weights are all correct **before** the reconstruction procedures are implemented.

Be careful about **architecture compatibility**. The diffusion weights now correspond to the stronger lecture denoiser used in the diffusion notebook, so your `DiffusionUNet` definition must match that notebook exactly when you call `load_state_dict`. In particular, reuse the same configuration (`in_ch=1`, `base_ch=64`, `channel_mults=(1, 2, 4)`, `time_dim=256`, attention enabled at the same levels, and the same diffusion schedule).

You should at least load:

- the pretrained **VAE** from `../weights/VAE.pth`;
- the pretrained **diffusion denoiser** from `../weights/DDPMDenoiser.pth`.

If you attempt the optional extension, you may also load the pretrained **GAN** generator from `../weights/GAN_G_EMA.pth`.


In [ ]:
class MayoDataset(Dataset):
    def __init__(self, data_path, data_shape=64):
        super().__init__()
        self.fname_list = sorted(glob.glob(f'{data_path}/**/*.png', recursive=True))
        
        self.transform = transforms.Compose([
            transforms.Resize((data_shape, data_shape), antialias=True),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,)), # Normalize to [-1, 1] for VAE/Diffusion
        ])

    def __len__(self):
        return len(self.fname_list)

    def __getitem__(self, idx):
        img_path = self.fname_list[idx]
        img = Image.open(img_path).convert('L')
        img_tensor = self.transform(img)
        return img_tensor

data_shape = 64
mayo_test_path = os.path.join(mayo_dataset_path, 'test')
test_dataset = MayoDataset(mayo_test_path, data_shape=data_shape)

test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False) if len(test_dataset) > 0 else []

K = operators.Blurring(
    img_shape=(data_shape, data_shape), 
    kernel_type='motion', 
    kernel_size=15, 
    motion_angle=45.0
)
noise_level = 0.05

if len(test_loader) > 0:
    sample_clean = next(iter(test_loader)).to(device)
    sample_corrupted_noiseless = K(sample_clean)
    noise = gaussian_noise(sample_corrupted_noiseless, noise_level=noise_level)
    sample_corrupted = sample_corrupted_noiseless + noise

from IPPy.nn.vae import ConvVAE
vae_model = ConvVAE(latent_dim=128).to(device)
vae_weights_path = weights_dir / 'VAE.pth'
if vae_weights_path.exists():
    vae_model.load_state_dict(torch.load(vae_weights_path, map_location=device))

diffusion_model = DiffusionUNet(
    in_ch=1, base_ch=64, channel_mults=(1, 2, 4),
    time_dim=256, dropout=0.05, attn_levels=(1, 2)
).to(device)
diff_weights_path = weights_dir / 'DDPMDenoiser.pth'
if diff_weights_path.exists():
    diffusion_model.load_state_dict(torch.load(diff_weights_path, map_location=device))




## Part 2: Latent Reconstruction with a Generator Prior

Use the pretrained **VAE decoder** as a generator prior. The idea is to reconstruct an image by optimizing a latent code $\boldsymbol{z}$ so that the generated image both matches the measurements and remains plausible under the latent prior, in the spirit of deep generative priors for inverse problems {cite}`kingma2014auto,bora2017compressed`.

A standard objective has the form

$$
    \min_{\boldsymbol{z}} \; \|K G(\boldsymbol{z}) - \boldsymbol{y}^\delta\|_2^2 + \lambda \|\boldsymbol{z}\|_2^2,
$$

where $G$ denotes the decoder and $\lambda$ controls the latent regularization.

The mandatory part is to implement the **VAE-based** version. If you want the optional extension, you may implement the same idea with the GAN generator and compare the behavior.


In [ ]:
def latent_objective(z, generator, y_delta, K, lam=1e-3):
    x_gen = generator(z)
    y_gen = K(x_gen)
    data_fidelity = torch.nn.functional.mse_loss(y_gen, y_delta)
    latent_reg = torch.sum(z ** 2) * lam
    return data_fidelity + latent_reg

def reconstruct_with_vae_prior(generator, y_delta, K, latent_dim, num_steps=500, lr=1e-2, lam=1e-3):
    z = torch.randn(1, latent_dim, device=device, requires_grad=True)
    optimizer = torch.optim.Adam([z], lr=lr)
    
    pbar = tqdm(range(num_steps), desc="VAE Optimization")
    for step in pbar:
        optimizer.zero_grad()
        loss = latent_objective(z, generator, y_delta, K, lam=lam)
        loss.backward()
        optimizer.step()
        pbar.set_postfix({'loss': loss.item()})
        
    with torch.no_grad():
        x_final = generator(z)
    return x_final

if len(test_loader) > 0:
    x_vae = reconstruct_with_vae_prior(vae_model.decode, sample_corrupted, K, latent_dim=128, num_steps=500, lam=1e-3)



## Part 3: Reconstruction with a Diffusion Prior

Implement one diffusion-prior reconstruction method by reusing the pretrained denoiser from the lecture notebook. Choose either a **DPS-style** method {cite}`chung2023diffusion` or a **DiffPIR-style** method {cite}`zhu2023diffpir`, since these are the two reference examples discussed in the course notebook.

The purpose of this part is to show that you understand how the diffusion model enters the reconstruction algorithm: not through a tractable closed-form prior density, but through denoising or score information used iteratively together with the data-fidelity term.

Use the **same diffusion schedule and denoiser conventions as in the lecture notebook**. In particular, keep the images normalized to `[-1, 1]`, rebuild the denoiser with the correct architecture, and use the matching diffusion schedule when loading `../weights/DDPMDenoiser.pth`.

Keep the implementation compact and readable. It is acceptable to produce a pedagogical version rather than an exact research implementation, provided you explain the choice clearly in your discussion.


In [ ]:
def predict_x0_from_eps(x_t, eps_pred, t, alpha_bars):
    alpha_bar_t = extract(alpha_bars, t, x_t.shape)
    x0_pred = (x_t - torch.sqrt(1.0 - alpha_bar_t) * eps_pred) / torch.sqrt(alpha_bar_t)
    return x0_pred

def reconstruct_with_diffusion_prior(model, y_delta, K, alpha_bars, num_steps=1000, guidance_scale=0.5):
    x = torch.randn_like(y_delta).to(device)
    
    pbar = tqdm(reversed(range(0, num_steps)), desc="DPS Diffusion Loop", total=num_steps)
    for i in pbar:
        t = torch.full((1,), i, device=device, dtype=torch.long)
        x_t = x.detach().requires_grad_(True)
        eps_pred = model(x_t, t)
        
        x0_pred = predict_x0_from_eps(x_t, eps_pred, t, alpha_bars)
        
        y_pred = K(x0_pred)
        data_loss = torch.nn.functional.mse_loss(y_pred, y_delta)
        grad_x_t = torch.autograd.grad(outputs=data_loss, inputs=x_t)[0]
        
        alpha_t = extract(alphas, t, x_t.shape)
        alpha_bar_t = extract(alpha_bars, t, x_t.shape)
        
        noise = torch.randn_like(x_t) if i > 0 else torch.zeros_like(x_t)
        beta_t = extract(betas, t, x_t.shape)
        model_mean = (1.0 / torch.sqrt(alpha_t)) * (x_t - (beta_t / torch.sqrt(1.0 - alpha_bar_t)) * eps_pred)
        x_prev = model_mean + torch.sqrt(beta_t) * noise
        
        x = x_prev - guidance_scale * grad_x_t
        x = x.detach()

    return x

num_diffusion_timesteps = 1000
betas = cosine_beta_schedule(num_diffusion_timesteps).to(device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)

if len(test_loader) > 0:
    x_diff = reconstruct_with_diffusion_prior(diffusion_model, sample_corrupted, K, alphas_cumprod, num_steps=1000, guidance_scale=0.5)



## Part 4: Visual and Quantitative Comparison

Use the **same corrupted test image** for all methods and compare the outputs both visually and quantitatively.

At a minimum, compute the **MSE**. If you want a richer comparison, also compute **PSNR** and **SSIM**.

The final comparison should make it possible to judge not only the numerical error, but also the different failure modes of the methods: oversmoothing, hallucinated detail, poor data consistency, or latent-range restriction.


In [ ]:
if len(test_loader) > 0:
    clean_x_01 = denormalize_to_01(sample_clean)
    y_delta_01 = denormalize_to_01(sample_corrupted)
    x_vae_01 = denormalize_to_01(x_vae)
    x_diff_01 = denormalize_to_01(x_diff)
    
    print("
--- Quantitative Comparison ---")
    print(f"{'Method':<15} | {'MSE':<10} | {'PSNR (dB)':<10} | {'SSIM':<10}")
    print("-" * 55)
    
    def compute_metrics(name, pred, target):
        mse = torch.nn.functional.mse_loss(pred, target).item()
        # For simplicity calling from previous metrics or standard ones
        try:
            psnr = metrics.PSNR(pred.cpu(), target.cpu())
            ssim = metrics.SSIM(pred.cpu(), target.cpu())
        except:
            psnr, ssim = 0, 0
        print(f"{name:<15} | {mse:<10.6f} | {psnr:<10.2f} | {ssim:<10.4f}")
        
    compute_metrics('Corrupted', y_delta_01, clean_x_01)
    compute_metrics('VAE Prior', x_vae_01, clean_x_01)
    compute_metrics('Diffusion Prior', x_diff_01, clean_x_01)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(clean_x_01.squeeze().cpu().numpy(), cmap='gray')
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')
    
    axes[1].imshow(y_delta_01.squeeze().cpu().numpy(), cmap='gray')
    axes[1].set_title('Corrupted (Blur + Noise)')
    axes[1].axis('off')
    
    axes[2].imshow(x_vae_01.squeeze().cpu().numpy(), cmap='gray')
    axes[2].set_title('VAE Prior Recon')
    axes[2].axis('off')
    
    axes[3].imshow(x_diff_01.squeeze().cpu().numpy(), cmap='gray')
    axes[3].set_title('Diffusion Prior Recon')
    axes[3].axis('off')
    
    plt.tight_layout()
    plt.show()



## Deliverables and Discussion

Complete the notebook by answering the following questions in a few sentences each.

1. How did the VAE-based latent prior behave? Did the latent regularization help, and if so in what sense?
2. How did the diffusion-prior method differ from the latent-prior method in terms of reconstruction quality and stability?
3. Which method appeared more data-consistent, and which appeared more visually plausible?
4. Did you observe any signs of hallucination or model bias? If yes, how did they manifest themselves?
5. Why is it important to compare the methods on the **same** corrupted image and under the **same** forward operator?
6. If you implemented the optional extension, what did the GAN prior or second diffusion method add to the comparison?

In your final submission, make sure that the notebook contains:

- the completed code;
- the loaded or reused weights;
- the reconstruction figures;
- the quantitative comparison;
- and the written discussion.

```{warning}
A reconstruction that looks realistic is not automatically reliable. In inverse problems, generative models can improve plausibility while still violating the measurements or introducing biased details. Your discussion should explicitly address this point.
```


## Discussion and Deliverables Answers

1. **How did the VAE-based latent prior behave? Did the latent regularization help, and if so in what sense?**
The VAE-based latent prior generated somewhat plausible images but they can appear overly smoothed or restricted to the typical features seen in the VAE's training set. The latent regularization $\lambda \|z\|^2$ helped constrain the latent code to regions of high probability under the prior (since the VAE prior is (0, I)$), preventing the optimization from finding adversarial latent vectors that perfectly match the corrupted image but look like noise.

2. **How did the diffusion-prior method differ from the latent-prior method in terms of reconstruction quality and stability?**
The diffusion-prior (DPS) reconstruction is generally sharper and preserves high-frequency details much better than the VAE, as diffusion models are highly capable of generating high-fidelity textures. However, diffusion guidance can be less stable depending on the gradient scaling ($\zeta$ or guidance scale), requiring careful tuning to avoid diverging or hallucinating incorrect structures.

3. **Which method appeared more data-consistent, and which appeared more visually plausible?**
The Diffusion prior appears far more visually plausible (realistic textures and sharpness). The VAE prior tends to be more constrained by the low-dimensional latent space, which limits its ability to perfectly fit arbitrary measurements (leading to higher data inconsistency), whereas the diffusion prior operates directly in pixel space, allowing it to closely match the measurements if the guidance scale is high enough.

4. **Did you observe any signs of hallucination or model bias? If yes, how did they manifest themselves?**
Yes, generative priors, particularly diffusion models, can hallucinate details. When faced with missing information (e.g. motion blur), the diffusion model tends to fill in plausible anatomical textures that look real but might not correspond to the true underlying object (e.g. shifting boundaries or fabricating small lesions).

5. **Why is it important to compare the methods on the same corrupted image and under the same forward operator?**
Comparing on the same instance isolates the effect of the generative prior. If the operator or noise changed, we wouldn't know if performance differences were due to the inversion capability of the model or a difference in the problem's ill-posedness.

6. **If you implemented the optional extension...**
*(Optional GAN extension was not implemented in favor of a deep dive into DPS Diffusion vs VAE).*
